Pneumonia X-Ray Classifier — ConvNeXt V2 Transfer Learning

| Feature | Detail |
|---|---|
| Model | ConvNeXt V2 Tiny (ImageNet-22k pretrained) |
| Input | RGB, ImageNet normalization, 224×224 |
| Strategy | Frozen head → full unfreeze at epoch 5 |
| Balancing | Class-weighted CrossEntropyLoss (auto-computed) |
| Split | Augmentation-aware group split zero leakage |
| Checkpoints | best.pt + latest.pt saved to Google Drive every epoch |
| Outputs | Training curves, confusion matrix, ROC curve, Grad-CAM |

> **Before running:** `Runtime → Change runtime type → T4 GPU`


Step 1 — Install Dependencies

In [3]:
!pip install -q timm grad-cam scikit-learn seaborn tqdm
import sys, torch
print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ No GPU — change runtime!'}")


Python  : 3.12.13
PyTorch : 2.10.0+cu128
GPU     : Tesla T4


Step 2 — Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted


Step 3 — Configuration

**Only edit this cell.**

Dataset structure expected:
```
your_dataset/
    NORMAL/      ← chest X-ray images
    PNEUMONIA/   ← chest X-ray images
```


In [ ]:
from pathlib import Path
import os

# ─── PATHS ────────────────────────────────────────────────────────────────────
DATA_DIR   = "/content/drive/MyDrive/ExecutionOutputs/pneumonia_dataset"   # ← YOUR dataset folder
OUTPUT_DIR = "/content/drive/MyDrive/ExecutionOutputs/pneumonia_outputs"   # ← where results are saved
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─── MODEL ────────────────────────────────────────────────────────────────────
MODEL_NAME = "convnextv2_base.fcmae_ft_in1k"

# ─── HYPERPARAMETERS ──────────────────────────────────────────────────────────
EPOCHS      = 30      # total training epochs
BATCH_SIZE  = 32      # reduce to 16 if GPU OOM
LR          = 3e-4    # learning rate for head-only phase
IMG_SIZE    = 224     # input resolution
UNFREEZE_EP = 5       # epoch to unfreeze full backbone (LR drops 10x)

# ─── DATA SPLITS ──────────────────────────────────────────────────────────────
# No manual split needed — dataset already has train/val/test folders
NUM_WORKERS = 2

# ─── MISC ─────────────────────────────────────────────────────────────────────
GRADCAM_SAMPLES = 8   # must be divisible by number of classes (2)
SEED            = 42

# ─── Verify Drive is mounted and create output dirs ───────────────────────────
if OUTPUT_DIR.startswith("/content/drive") and not os.path.exists("/content/drive/MyDrive"):
    raise RuntimeError("Google Drive not mounted — run Step 2 first")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR, "checkpoints").mkdir(parents=True, exist_ok=True)

print("Config ready")
print(f"   Data      : {DATA_DIR}")
print(f"   Outputs   : {OUTPUT_DIR}")
print(f"   Model     : {MODEL_NAME}")
print(f"   Epochs    : {EPOCHS}  |  Unfreeze at: {UNFREEZE_EP}  |  LR: {LR}")


✅ Config ready
   Data      : /content/drive/MyDrive/ExecutionOutputs/pneumonia_dataset
   Outputs   : /content/drive/MyDrive/ExecutionOutputs/pneumonia_outputs
   Model     : convnextv2_base.fcmae_ft_in1k
   Epochs    : 30  |  Unfreeze at: 5  |  LR: 0.0003


Step 4 — Imports & Reproducibility

In [ ]:
import os, re, random, time, warnings
from pathlib import Path
from collections import defaultdict

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
from sklearn.utils.class_weight import compute_class_weight
import timm
from tqdm.notebook import tqdm

try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    GRADCAM_AVAILABLE = True
    print("Grad-CAM available")
except ImportError:
    GRADCAM_AVAILABLE = False
    print("Grad-CAM not found — re-run Step 1")

warnings.filterwarnings("ignore")

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}" + (f"  ({torch.cuda.get_device_name(0)})" if DEVICE.type=='cuda' else ""))


✅ Grad-CAM available
✅ Device : cuda  (Tesla T4)


Step 5 — Data Loading & Augmentation-Aware Split

Splits at the **root image group level** — all `_aug_N` variants of the same original always stay in the same split. A leakage report is printed every run.


In [ ]:
# ── Diagnose dataset folder structure ────────────────────────────────────────
import os
from pathlib import Path
from collections import defaultdict

print(f"Scanning: {DATA_DIR}\n")

for split in ["train", "val", "test"]:
    split_dir = Path(DATA_DIR) / split
    if not split_dir.exists():
        print(f"Missing folder: {split_dir}")
        continue
    print(f"📁 {split}/")
    for cls_dir in sorted(split_dir.iterdir()):
        if not cls_dir.is_dir():
            continue
        files     = list(cls_dir.iterdir())
        ext_counts = defaultdict(int)
        for f in files:
            ext_counts[f.suffix.lower()] += 1
        ext_str = "  ".join(f"{ext}: {count}" for ext, count in sorted(ext_counts.items()))
        print(f"   {cls_dir.name}/  ({len(files)} files)   [{ext_str if ext_str else 'EMPTY'}]")
    print()


Scanning: /content/drive/MyDrive/ExecutionOutputs/pneumonia_dataset

📁 train/
   Normal/  (2991 files)   [.png: 2991]
   Pneumonia/  (2991 files)   [.png: 2991]

📁 val/
   Normal/  (237 files)   [.png: 237]
   Pneumonia/  (641 files)   [.png: 641]

📁 test/
   Normal/  (238 files)   [.png: 238]
   Pneumonia/  (641 files)   [.png: 641]



In [ ]:
def get_transforms(img_size, is_train=True):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    # if is_train:
    #     return transforms.Compose([
    #         transforms.Resize((img_size + 32, img_size + 32)),
    #         transforms.RandomCrop(img_size),
    #         transforms.RandomHorizontalFlip(),
    #         transforms.RandomRotation(10),
    #         transforms.ColorJitter(brightness=0.2, contrast=0.2),
    #         transforms.ToTensor(),
    #         transforms.Normalize(mean, std),
    #     ])
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])


def build_dataloaders(data_dir, img_size, batch_size, num_workers):
    """
    Reads pre-split dataset directly from existing folder structure:
        data_dir/train/NORMAL
        data_dir/train/PNEUMONIA
        data_dir/val/NORMAL
        data_dir/val/PNEUMONIA
        data_dir/test/NORMAL
        data_dir/test/PNEUMONIA
    No manual splitting needed — uses the provided splits as-is.
    """
    train_dir = Path(data_dir) / "train"
    val_dir   = Path(data_dir) / "val"
    test_dir  = Path(data_dir) / "test"

    # Verify all folders exist
    for d in [train_dir, val_dir, test_dir]:
        if not d.exists():
            raise FileNotFoundError(f"Split folder not found: {d}")

    # Accept uppercase extensions (.JPEG, .PNG, .JPG) as well as lowercase
    VALID_EXTS = {".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".tif", ".tiff", ".webp"}
    def is_valid(path): return Path(path).suffix.lower() in VALID_EXTS

    train_ds = datasets.ImageFolder(train_dir, transform=get_transforms(img_size, True),  is_valid_file=is_valid)
    val_ds   = datasets.ImageFolder(val_dir,   transform=get_transforms(img_size, False), is_valid_file=is_valid)
    test_ds  = datasets.ImageFolder(test_dir,  transform=get_transforms(img_size, False), is_valid_file=is_valid)

    # Verify class names are consistent across splits
    assert train_ds.classes == val_ds.classes == test_ds.classes, \
        f"Class mismatch across splits: {train_ds.classes} vs {val_ds.classes} vs {test_ds.classes}"

    class_names = train_ds.classes

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size,
                              shuffle=False, num_workers=num_workers, pin_memory=True)

    # ── Print split summary ───────────────────────────────────────────────────
    def class_dist(ds):
        from collections import Counter
        counts = Counter(ds.targets)
        return {class_names[k]: v for k, v in sorted(counts.items())}

    print(f"{'─'*54}")
    print(f"  Classes : {class_names}")
    for name, ds in [("Train", train_ds), ("Val", val_ds), ("Test", test_ds)]:
        d    = class_dist(ds)
        dist = "  |  ".join(f"{k}: {v}" for k, v in d.items())
        print(f"  {name:<6}: {len(ds):>5} images   [{dist}]")
    print(f"{'─'*54}\n")

    return train_loader, val_loader, test_loader, class_names, test_ds


train_loader, val_loader, test_loader, CLASS_NAMES, test_subset = build_dataloaders(
    DATA_DIR, IMG_SIZE, BATCH_SIZE, NUM_WORKERS
)
NUM_CLASSES = len(CLASS_NAMES)

# IDX_TRAIN for class weight computation — all training indices
IDX_TRAIN = list(range(len(train_loader.dataset)))


──────────────────────────────────────────────────────
  Classes : ['Normal', 'Pneumonia']
  Train :  5982 images   [Normal: 2991  |  Pneumonia: 2991]
  Val   :   878 images   [Normal: 237  |  Pneumonia: 641]
  Test  :   879 images   [Normal: 238  |  Pneumonia: 641]
──────────────────────────────────────────────────────



Preview Sample Training Images

In [ ]:
mean_np = np.array([0.485, 0.456, 0.406])
std_np  = np.array([0.229, 0.224, 0.225])
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
fig.suptitle("Sample training images (after augmentation)", fontsize=12, fontweight="bold")
for i, ax in enumerate(axes.flat):
    if i >= len(images): break
    img = np.clip(images[i].permute(1,2,0).numpy() * std_np + mean_np, 0, 1)
    ax.imshow(img); ax.set_title(CLASS_NAMES[labels[i]], fontsize=8); ax.axis("off")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/sample_images.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: sample_images.png")


✅ Saved: sample_images.png


Step 6 — Model + Class Weights

ConvNeXt V2 Tiny pretrained on ImageNet-22k. Backbone is frozen for the first `UNFREEZE_EP` epochs — only the classification head trains. Class weights are auto-computed from training labels to correct any residual imbalance.


In [ ]:
def build_model(model_name, num_classes, device):
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
    for name, param in model.named_parameters():
        if "head" not in name:
            param.requires_grad = False
    model = model.to(device)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  Model      : {model_name}")
    print(f"  Parameters : {total:,} total  |  {trainable:,} trainable (head only)")
    return model


def unfreeze_backbone(model, lr):
    for p in model.parameters():
        p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    new_opt   = optim.AdamW(model.parameters(), lr=lr * 0.1, weight_decay=1e-4)
    print(f"\nBackbone unfrozen — {trainable:,} params  |  fine-tune LR: {lr*0.1:.2e}")
    return new_opt


model = build_model(MODEL_NAME, NUM_CLASSES, DEVICE)

# ── Class-weighted loss ────────────────────────────────────────────────────────
_full_ds_tmp  = datasets.ImageFolder(DATA_DIR)
train_labels  = [_full_ds_tmp.samples[i][1] for i in IDX_TRAIN]
class_weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=train_labels)
weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)

w_str = "  |  ".join(f"{CLASS_NAMES[i]}: {w:.4f}x" for i, w in enumerate(class_weights))
print(f"\n  Class weights : [{w_str}]")
print("  (weights near 1.0 = dataset already balanced)")

criterion = nn.CrossEntropyLoss(weight=weights_tensor, label_smoothing=0.1)
optimizer  = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                         lr=LR, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
print("\nModel, weighted loss, optimizer and scheduler ready")


model.safetensors:   0%|          | 0.00/355M [00:00<?, ?B/s]

  Model      : convnextv2_base.fcmae_ft_in1k
  Parameters : 87,694,850 total  |  4,098 trainable (head only)

  Class weights : [Normal: 3.4027x  |  Pneumonia: 0.5861x]
  (weights near 1.0 = dataset already balanced)

✅ Model, weighted loss, optimizer and scheduler ready


Step 7 — Checkpoints

Auto-resumes from `latest.pt` if the session was interrupted.

In [ ]:
CKPT_DIR = Path(OUTPUT_DIR) / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)


def save_checkpoint(state, path):
    torch.save(state, path)


def load_checkpoint(path, model, optimizer=None, scheduler=None):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    if optimizer and "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    if scheduler and "scheduler_state" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    return ckpt.get("epoch", 0), ckpt.get("best_val_auc", 0.0)


latest_ckpt  = CKPT_DIR / "latest.pt"
start_epoch  = 0
best_val_auc = 0.0

if latest_ckpt.exists():
    # ── Verify checkpoint matches current model before loading ────────────────
    ckpt = torch.load(latest_ckpt, map_location="cpu", weights_only=False)
    saved_model = ckpt.get("model_name", "unknown")
    current_model = MODEL_NAME

    if saved_model != current_model:
        print(f"   Checkpoint mismatch!")
        print(f"   Saved model    : {saved_model}")
        print(f"   Current model  : {current_model}")
        print(f"   Deleting stale checkpoint and starting fresh...")
        latest_ckpt.unlink()                          # delete latest.pt
        best_ckpt = CKPT_DIR / "best.pt"
        if best_ckpt.exists():
            best_ckpt.unlink()                        # delete best.pt too
        print("Stale checkpoints removed — starting fresh")
    else:
        print(f"Resuming from checkpoint ({saved_model})...")
        start_epoch, best_val_auc = load_checkpoint(latest_ckpt, model, optimizer, scheduler)
        model.to(DEVICE)
        print(f"   Epoch {start_epoch}  |  best AUC so far: {best_val_auc:.4f}")
else:
    print("No checkpoint found — starting fresh")

🆕 No checkpoint found — starting fresh


Step 8 — Training Loop

Backbone unfreezes at epoch `UNFREEZE_EP` with LR dropped 10×. If Colab disconnects, re-run Steps 7 → 8 — auto-resumes from `latest.pt`.


In [ ]:
def run_epoch(model, loader, criterion, optimizer, device, is_train):
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    ctx   = torch.enable_grad() if is_train else torch.no_grad()
    label = "Train" if is_train else "Val  "
    with ctx:
        for images, labels in tqdm(loader, desc=f"  {label}", leave=False):
            images, labels = images.to(device), labels.to(device)
            if is_train: optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            if is_train: loss.backward(); optimizer.step()
            total_loss += loss.item() * images.size(0)
            probs   = torch.softmax(outputs, dim=1)
            preds   = probs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += images.size(0)
            all_probs.extend(probs.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())
    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    auc = roc_auc_score(all_labels, all_probs[:,1]) if all_probs.shape[1]==2           else roc_auc_score(all_labels, all_probs, multi_class="ovr")
    return total_loss / total, correct / total, auc


history = {k: [] for k in ["train_loss","val_loss","train_acc","val_acc","train_auc","val_auc"]}

print(f"{'═'*62}")
print(f"  Training  |  epochs: {EPOCHS}  |  unfreeze backbone at: epoch {UNFREEZE_EP}")
print(f"{'═'*62}\n")

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()

    if epoch == UNFREEZE_EP:
        optimizer = unfreeze_backbone(model, LR)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS - epoch, eta_min=1e-7)

    tr_loss, tr_acc, tr_auc = run_epoch(model, train_loader, criterion, optimizer, DEVICE, True)
    vl_loss, vl_acc, vl_auc = run_epoch(model, val_loader,   criterion, None,      DEVICE, False)
    scheduler.step()

    for k, v in zip(history, [tr_loss, vl_loss, tr_acc, vl_acc, tr_auc, vl_auc]):
        history[k].append(v)

    star = "⭐" if vl_auc > best_val_auc else "  "
    print(f"{star} [{epoch+1:03d}/{EPOCHS}]  "
          f"Loss {tr_loss:.4f}/{vl_loss:.4f}  "
          f"Acc {tr_acc*100:.1f}%/{vl_acc*100:.1f}%  "
          f"AUC {tr_auc:.4f}/{vl_auc:.4f}  "
          f"LR {optimizer.param_groups[0]['lr']:.1e}  "
          f"{time.time()-t0:.0f}s")

    ckpt = dict(epoch=epoch+1,
                model_state=model.state_dict(),
                optimizer_state=optimizer.state_dict(),
                scheduler_state=scheduler.state_dict(),
                best_val_auc=best_val_auc,
                class_names=CLASS_NAMES,
                model_name=MODEL_NAME)
    save_checkpoint(ckpt, CKPT_DIR / "latest.pt")

    if vl_auc > best_val_auc:
        best_val_auc = vl_auc
        ckpt["best_val_auc"] = best_val_auc
        save_checkpoint(ckpt, CKPT_DIR / "best.pt")

print(f"\nTraining done  |  Best val AUC: {best_val_auc:.4f}")


══════════════════════════════════════════════════════════════
  Training  |  epochs: 30  |  unfreeze backbone at: epoch 5
══════════════════════════════════════════════════════════════



  Train:   0%|          | 0/187 [00:00<?, ?it/s]

  Val  :   0%|          | 0/28 [00:00<?, ?it/s]

⭐ [001/30]  Loss 0.3422/0.8468  Acc 58.1%/57.2%  AUC 0.8750/0.9436  LR 3.0e-04  1436s


  Train:   0%|          | 0/187 [00:00<?, ?it/s]

...

  Val  :   0%|          | 0/28 [00:00<?, ?it/s]

   [029/30]  Loss 0.1451/0.5293  Acc 100.0%/98.5%  AUC 1.0000/0.9965  LR 2.2e-07  344s


  Train:   0%|          | 0/187 [00:00<?, ?it/s]

  Val  :   0%|          | 0/28 [00:00<?, ?it/s]

   [030/30]  Loss 0.1456/0.5293  Acc 100.0%/98.5%  AUC 1.0000/0.9965  LR 1.0e-07  344s

✅ Training done  |  Best val AUC: 0.9980


Step 9 — Training Curves

In [ ]:
ep = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"Training history — {MODEL_NAME}", fontsize=13, fontweight="bold")
for ax, (tk, vk), title, color in zip(
    axes,
    [("train_loss","val_loss"),("train_acc","val_acc"),("train_auc","val_auc")],
    ["Loss","Accuracy","AUC"],
    ["#e74c3c","#2ecc71","#3498db"]
):
    ax.plot(ep, history[tk], label="Train", color=color, linewidth=2)
    ax.plot(ep, history[vk], label="Val",   color=color, linewidth=2, linestyle="--")
    if title == "Accuracy":
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f"{y*100:.0f}%"))
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_curves.png")

# ── CSV export: training history ─────────────────────────────────────────────
import pandas as pd
history_df = pd.DataFrame({
    "epoch":      list(ep),
    "train_loss": history["train_loss"],
    "val_loss":   history["val_loss"],
    "train_acc":  history["train_acc"],
    "val_acc":    history["val_acc"],
    "train_auc":  history["train_auc"],
    "val_auc":    history["val_auc"],
})
history_csv = f"{OUTPUT_DIR}/training_history.csv"
history_df.to_csv(history_csv, index=False)
print(f"Saved CSV: training_history.csv")
print(history_df.to_string(index=False))


📈 Saved: training_curves.png
📄 Saved CSV: training_history.csv
 epoch  train_loss  val_loss  train_acc  val_acc  train_auc  val_auc
     1    0.342187  0.846810   0.581244 0.571754   0.875001 0.943594
     2    0.247179  0.754858   0.762287 0.685649   0.971331 0.960946
     3    0.227835  0.713409   0.800736 0.749431   0.980475 0.970385
     4    0.218151  0.702567   0.825811 0.764237   0.984893 0.975368
     5    0.211774  0.681449   0.836175 0.785877   0.987385 0.978106
     6    0.188305  0.538851   0.923103 0.971526   0.991436 0.996465
     7    0.162024  0.566331   0.970913 0.940774   0.998608 0.997360
     8    0.151927  0.529392   0.987797 0.979499   0.999473 0.997400
     9    0.149570  0.532455   0.992979 0.978360   0.999805 0.995465
    10    0.147504  0.538946   0.994985 0.976082   0.999981 0.992647
    11    0.145209  0.533759   0.999833 0.974943   1.000000 0.994800
    12    0.145288  0.527440   0.999833 0.984055   1.000000 0.994583
    13    0.144851  0.524018   0.999833 

Step 10 — Test Set Evaluation

In [ ]:
print("Loading best checkpoint...")
load_checkpoint(CKPT_DIR / "best.pt", model)
model.to(DEVICE).eval()

all_preds, all_probs_list, all_labels_list = [], [], []
total_loss, correct, total = 0.0, 0, 0

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="  Test"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss    = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        probs   = torch.softmax(outputs, dim=1)
        preds   = probs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += images.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_probs_list.extend(probs.cpu().numpy())
        all_labels_list.extend(labels.cpu().numpy())

all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs_list)
all_labels = np.array(all_labels_list)

test_acc = correct / total
test_f1  = f1_score(all_labels, all_preds, average="weighted")
test_auc = roc_auc_score(all_labels, all_probs[:,1]) if NUM_CLASSES==2            else roc_auc_score(all_labels, all_probs, multi_class="ovr")

print(f"\n{'═'*52}")
print(f"  TEST RESULTS")
print(f"{'═'*52}")
print(f"  Accuracy  : {test_acc*100:.2f}%")
print(f"  AUC       : {test_auc:.4f}")
print(f"  F1        : {test_f1:.4f}")
print(f"  Loss      : {total_loss/total:.4f}")
print(f"{'═'*52}")
print(f"\n{classification_report(all_labels, all_preds, target_names=CLASS_NAMES)}")

# ── CSV export: overall test metrics ──────────────────────────────────────────
import pandas as pd
metrics_df = pd.DataFrame([{
    "model":        MODEL_NAME,
    "test_accuracy": round(test_acc, 6),
    "test_auc":      round(test_auc, 6),
    "test_f1":       round(test_f1, 6),
    "test_loss":     round(total_loss / total, 6),
    "n_samples":     total,
}])
metrics_csv = f"{OUTPUT_DIR}/test_metrics.csv"
metrics_df.to_csv(metrics_csv, index=False)
print(f"Saved CSV: test_metrics.csv")

# ── CSV export: per-class classification report ────────────────────────────────
from sklearn.metrics import classification_report
report_dict = classification_report(
    all_labels, all_preds, target_names=CLASS_NAMES, output_dict=True
)
report_df = pd.DataFrame(report_dict).transpose().reset_index()
report_df.rename(columns={"index": "class"}, inplace=True)
report_csv = f"{OUTPUT_DIR}/classification_report.csv"
report_df.to_csv(report_csv, index=False)
print(f"Saved CSV: classification_report.csv")
print(report_df.to_string(index=False))


Loading best checkpoint...


  Test:   0%|          | 0/28 [00:00<?, ?it/s]


════════════════════════════════════════════════════
  TEST RESULTS
════════════════════════════════════════════════════
  Accuracy  : 97.84%
  AUC       : 0.9971
  F1        : 0.9785
  Loss      : 0.5368
════════════════════════════════════════════════════

              precision    recall  f1-score   support

      Normal       0.94      0.98      0.96       238
   Pneumonia       0.99      0.98      0.99       641

    accuracy                           0.98       879
   macro avg       0.97      0.98      0.97       879
weighted avg       0.98      0.98      0.98       879

📄 Saved CSV: test_metrics.csv
📄 Saved CSV: classification_report.csv
       class  precision   recall  f1-score    support
      Normal   0.943320 0.978992  0.960825 238.000000
   Pneumonia   0.992089 0.978159  0.985075 641.000000
    accuracy   0.978385 0.978385  0.978385   0.978385
   macro avg   0.967704 0.978575  0.972950 879.000000
weighted avg   0.978884 0.978385  0.978509 879.000000


Confusion Matrix

In [ ]:
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, fmt, title in zip(
    axes, [cm, cm_norm], ["d", ".2f"],
    ["Counts", "Normalized"]
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=0.5)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion matrix — {title}", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrix.png")

# ── CSV export: confusion matrix (counts) ────────────────────────────────────
import pandas as pd
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_df.index.name = "true \ predicted"
cm_csv = f"{OUTPUT_DIR}/confusion_matrix.csv"
cm_df.to_csv(cm_csv)
print(f"Saved CSV: confusion_matrix.csv")
print(cm_df.to_string())

# ── CSV export: confusion matrix (normalized) ─────────────────────────────────
cm_norm_df = pd.DataFrame(cm_norm.round(4), index=CLASS_NAMES, columns=CLASS_NAMES)
cm_norm_df.index.name = "true \ predicted"
cm_norm_csv = f"{OUTPUT_DIR}/confusion_matrix_normalized.csv"
cm_norm_df.to_csv(cm_norm_csv)
print(f"Saved CSV: confusion_matrix_normalized.csv")


📊 Saved: confusion_matrix.png
📄 Saved CSV: confusion_matrix.csv
                  Normal  Pneumonia
true \ predicted                   
Normal               233          5
Pneumonia             14        627
📄 Saved CSV: confusion_matrix_normalized.csv


ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
if NUM_CLASSES == 2:
    fpr, tpr, _ = roc_curve(all_labels, all_probs[:,1])
    ax.plot(fpr, tpr, linewidth=2.5, color="#2ecc71",
            label=f"AUC = {roc_auc_score(all_labels, all_probs[:,1]):.4f}")
else:
    for i, (name, color) in enumerate(zip(CLASS_NAMES, plt.cm.tab10.colors)):
        fpr, tpr, _ = roc_curve((all_labels==i).astype(int), all_probs[:,i])
        auc_i = roc_auc_score((all_labels==i).astype(int), all_probs[:,i])
        ax.plot(fpr, tpr, linewidth=2, color=color, label=f"{name}  AUC={auc_i:.3f}")
ax.plot([0,1],[0,1],"k--",linewidth=1,label="Random")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("ROC curve", fontweight="bold")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: roc_curve.png")

# ── CSV export: ROC curve data points ─────────────────────────────────────────
import pandas as pd
roc_rows = []
if NUM_CLASSES == 2:
    fpr_vals, tpr_vals, thresholds = roc_curve(all_labels, all_probs[:,1])
    for f, t, th in zip(fpr_vals, tpr_vals, thresholds):
        roc_rows.append({"class": CLASS_NAMES[1], "fpr": round(f, 6),
                         "tpr": round(t, 6), "threshold": round(float(th), 6)})
else:
    for i, name in enumerate(CLASS_NAMES):
        fpr_vals, tpr_vals, thresholds = roc_curve((all_labels==i).astype(int), all_probs[:,i])
        for f, t, th in zip(fpr_vals, tpr_vals, thresholds):
            roc_rows.append({"class": name, "fpr": round(f, 6),
                             "tpr": round(t, 6), "threshold": round(float(th), 6)})
roc_df = pd.DataFrame(roc_rows)
roc_csv = f"{OUTPUT_DIR}/roc_curve.csv"
roc_df.to_csv(roc_csv, index=False)
print(f"Saved CSV: roc_curve.csv  ({len(roc_df)} data points)")
print(roc_df.head(10).to_string(index=False))


📉 Saved: roc_curve.png
📄 Saved CSV: roc_curve.csv  (34 data points)
    class      fpr      tpr  threshold
Pneumonia 0.000000 0.000000        inf
Pneumonia 0.000000 0.001560   0.783546
Pneumonia 0.000000 0.006240   0.782003
Pneumonia 0.000000 0.009360   0.780938
Pneumonia 0.000000 0.790952   0.761570
Pneumonia 0.000000 0.794072   0.761555
Pneumonia 0.000000 0.939158   0.742150
Pneumonia 0.004202 0.939158   0.742021
Pneumonia 0.004202 0.946958   0.732463
Pneumonia 0.008403 0.946958   0.731871


Step 11 — Grad-CAM Visualizations

Balanced sampling — exactly `GRADCAM_SAMPLES // 2` images per class. Green title = correct prediction, Red = wrong.


In [ ]:
if not GRADCAM_AVAILABLE:
    print("Grad-CAM not available — re-run Step 1 and restart kernel")
else:
    # ── Find last Conv2d layer dynamically ────────────────────────────────────
    target_layers  = []
    last_conv_name = ""
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            target_layers  = [module]
            last_conv_name = name
    if not target_layers:
        print("No Conv2d layer found — cannot run Grad-CAM")
    else:
        print(f"Grad-CAM target layer: {last_conv_name}")

        # ── Balanced class sampling ───────────────────────────────────────────
        class_indices = defaultdict(list)
        for idx in range(len(test_subset)):
            _, lbl = test_subset[idx]
            class_indices[lbl].append(idx)

        n_per_class = GRADCAM_SAMPLES // NUM_CLASSES
        indices = []
        for cls in sorted(class_indices.keys()):
            chosen = random.sample(class_indices[cls], min(n_per_class, len(class_indices[cls])))
            indices.extend(chosen)
            print(f"  Sampling {len(chosen)} images from: {CLASS_NAMES[cls]}")
        random.shuffle(indices)
        n = len(indices)

        # ── Build CAM and plot ────────────────────────────────────────────────
        cam = GradCAM(model=model, target_layers=target_layers)
        fig, axes = plt.subplots(2, n, figsize=(n * 3, 6))
        fig.suptitle("Grad-CAM  |  Row 1: original   Row 2: activation heatmap",
                     fontsize=12, fontweight="bold")

        for col, idx in enumerate(indices):
            img_t, label = test_subset[idx]
            inp = img_t.unsqueeze(0).to(DEVICE)
            inp.requires_grad_(True)       # Grad-CAM needs gradient tracking

            model.eval()                   # eval mode — no no_grad wrapper
            pred = model(inp).argmax(1).item()

            gcam = cam(input_tensor=inp, targets=[ClassifierOutputTarget(pred)])[0]

            # Denormalize for display
            mean_np = np.array([0.485, 0.456, 0.406])
            std_np  = np.array([0.229, 0.224, 0.225])
            img_np  = img_t.permute(1,2,0).numpy()
            img_np  = np.clip(img_np * std_np + mean_np, 0, 1).astype(np.float32)
            cam_img = show_cam_on_image(img_np, gcam, use_rgb=True)

            axes[0, col].imshow(img_np)
            axes[0, col].set_title(f"True:\n{CLASS_NAMES[label]}", fontsize=8)
            axes[0, col].axis("off")

            axes[1, col].imshow(cam_img)
            axes[1, col].set_title(f"Pred:\n{CLASS_NAMES[pred]}", fontsize=8,
                                   color="green" if pred==label else "red")
            axes[1, col].axis("off")


        # ── CSV export: Grad-CAM per-image predictions ───────────────────────────
        import pandas as pd
        gradcam_records = []
        for col, idx in enumerate(indices):
            img_t2, label2 = test_subset[idx]
            inp2 = img_t2.unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                pred2 = model(inp2).argmax(1).item()
                prob2 = torch.softmax(model(inp2), dim=1)[0].cpu().numpy()
            gradcam_records.append({
                "sample_index":   idx,
                "true_label":     CLASS_NAMES[label2],
                "pred_label":     CLASS_NAMES[pred2],
                "correct":        label2 == pred2,
                **{f"prob_{CLASS_NAMES[c]}": round(float(prob2[c]), 6) for c in range(NUM_CLASSES)},
            })
        gradcam_df = pd.DataFrame(gradcam_records)
        gradcam_csv = f"{OUTPUT_DIR}/gradcam_predictions.csv"
        gradcam_df.to_csv(gradcam_csv, index=False)
        print(f"Saved CSV: gradcam_predictions.csv  ({len(gradcam_df)} samples)")
        print(gradcam_df.to_string(index=False))
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/gradcam.png", dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved: gradcam.png  ({n_per_class} per class)")


✅ Grad-CAM target layer: stages.3.blocks.2.conv_dw
  Sampling 4 images from: Normal
  Sampling 4 images from: Pneumonia
📄 Saved CSV: gradcam_predictions.csv  (8 samples)
 sample_index true_label pred_label  correct  prob_Normal  prob_Pneumonia
          189     Normal     Normal     True     0.991699        0.008301
          380  Pneumonia  Pneumonia     True     0.233205        0.766795
            6     Normal     Normal     True     0.944127        0.055873
          163     Normal     Normal     True     0.897656        0.102344
          519  Pneumonia  Pneumonia     True     0.236248        0.763752
          466  Pneumonia  Pneumonia     True     0.234097        0.765903
          488  Pneumonia  Pneumonia     True     0.233778        0.766222
           28     Normal     Normal     True     0.990976        0.009024
🔥 Saved: gradcam.png  (4 per class)


Step 12 — Summary

In [ ]:
from IPython.display import display, HTML

html = f"""
<div style="font-family:monospace;background:var(--jp-layout-color1,#f8f9fa);
            padding:20px;border-radius:8px;border-left:4px solid #2ecc71">
  <h3 style="margin-top:0;color:#2c3e50">Training complete</h3>
  <table style="width:100%;border-collapse:collapse;font-size:14px">
    <tr><td style="padding:5px;color:#666">Model</td>
        <td><b>{MODEL_NAME}</b></td></tr>
    <tr><td style="padding:5px;color:#666">Test accuracy</td>
        <td style="color:#e74c3c;font-weight:bold">{test_acc*100:.2f}%</td></tr>
    <tr><td style="padding:5px;color:#666">Test AUC</td>
        <td style="color:#3498db;font-weight:bold">{test_auc:.4f}</td></tr>
    <tr><td style="padding:5px;color:#666">Test F1</td>
        <td style="color:#9b59b6;font-weight:bold">{test_f1:.4f}</td></tr>
    <tr><td style="padding:5px;color:#666">Best val AUC</td>
        <td>{best_val_auc:.4f}</td></tr>
    <tr><td style="padding:5px;color:#666">Classes</td>
        <td>{CLASS_NAMES}</td></tr>
  </table>
  <br>
  <b>Saved to:</b> {OUTPUT_DIR}/<br>
  <ul style="margin:6px 0;padding-left:20px;font-size:13px">
    <li>checkpoints/best.pt — best model weights</li>
    <li>checkpoints/latest.pt — resume token</li>
    <li>sample_images.png | training_curves.png</li>
    <li>confusion_matrix.png | roc_curve.png | gradcam.png</li>
    <li>training_history.csv | test_metrics.csv | classification_report.csv</li>
    <li>confusion_matrix.csv | confusion_matrix_normalized.csv</li>
    <li>roc_curve.csv | gradcam_predictions.csv | summary.csv</li>
  </ul>
</div>
"""
display(HTML(html))

# ── CSV export: summary of the full run ───────────────────────────────────────
import pandas as pd
summary_df = pd.DataFrame([{
    "model":          MODEL_NAME,
    "classes":        str(CLASS_NAMES),
    "test_accuracy":  round(test_acc, 6),
    "test_auc":       round(test_auc, 6),
    "test_f1":        round(test_f1, 6),
    "best_val_auc":   round(best_val_auc, 6),
    "output_dir":     OUTPUT_DIR,
}])
summary_csv = f"{OUTPUT_DIR}/summary.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"Saved CSV: summary.csv")
print(summary_df.to_string(index=False))


Model,convnextv2_base.fcmae_ft_in1k
Test accuracy,97.84%
Test AUC,0.9971
Test F1,0.9785
Best val AUC,0.9980
Classes,"['Normal', 'Pneumonia']"


📄 Saved CSV: summary.csv
                        model                 classes  test_accuracy  test_auc  test_f1  best_val_auc                                                output_dir
convnextv2_base.fcmae_ft_in1k ['Normal', 'Pneumonia']       0.978385   0.99709 0.978509      0.997973 /content/drive/MyDrive/ExecutionOutputs/pneumonia_outputs
